In [ ]:
# sklearn_runner.py (or notebook cell)
import numpy as np, pandas as pd
from pathlib import Path
from features import get_feature
from models import get_model            # knn1, rf
from eval_utils import subject_folds, enroll_per_subject, accuracy, far_frr, mean_eer_ovr

PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"
OUT = Path("results/nxk_sklearn.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)

FEATURE_SET = ["simple", "dwt_db4_l2", "spectrum_raw"]
MODEL_SET = ["knn1", "rf"]
N_SPLITS, SEED, ENROLL_K = 5, 42, 5

X_raw = pd.read_csv(PROC_CSV).values
y = pd.read_csv(LABS_CSV)["subject_id"].values
groups = y.copy()

rows = []
for f_name in FEATURE_SET:
    X_feat = X_raw if f_name == "spectrum_raw" else get_feature(f_name).fit(X_raw, y).transform(X_raw)
    for fold, (tr, te) in enumerate(subject_folds(X_feat, y, groups, n_splits=N_SPLITS, seed=SEED), 1):
        X_tr, y_tr = X_feat[tr], y[tr]
        X_te, y_te = X_feat[te], y[te]
        X_en, y_en, X_q, y_q = enroll_per_subject(X_te, y_te, k=ENROLL_K)
        X_tr2 = np.vstack([X_tr, X_en]) if len(X_en) else X_tr
        y_tr2 = np.concatenate([y_tr, y_en]) if len(y_en) else y_tr
        for m_name in MODEL_SET:
            try:
                model = get_model(m_name)   # sklearn only
                model.fit(X_tr2, y_tr2)
                yhat = model.predict(X_q)
                A = accuracy(y_q, yhat)
                FAR, FRR = far_frr(y_q, yhat)
                EER = np.nan
                if hasattr(model, "predict_proba"):
                    try:
                        yscore = model.predict_proba(X_q)
                        if yscore is not None:
                            EER = mean_eer_ovr(y_q, yscore, np.unique(y_q))
                    except Exception:
                        pass
                rows.append({"feature": f_name, "model": m_name, "fold": fold,
                             "accuracy": A, "far": FAR, "frr": FRR, "eer": EER, "error": np.nan})
            except Exception as e:
                rows.append({"feature": f_name, "model": m_name, "fold": fold,
                             "accuracy": np.nan, "far": np.nan, "frr": np.nan, "eer": np.nan, "error": str(e)})

df = pd.DataFrame(rows)
df.to_csv(OUT, index=False)
print(f"Saved → {OUT}")


In [ ]:
# dwt_mlp_runner.py with training logs per fold

import os
import numpy as np, pandas as pd, torch
from pathlib import Path
from features import get_feature
from models import get_model            # dwt_mlp
from eval_utils import subject_folds, enroll_per_subject, accuracy, far_frr, mean_eer_ovr

# Optional: precise CUDA stack traces (enable only for debugging)
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"
OUT = Path("results/nxk_dwt_mlp.csv")
LOG_OUT = Path("results/nxk_dwt_mlp_logs.txt")
OUT.parent.mkdir(parents=True, exist_ok=True)

FEATURE = "dwt_db4_l2"
MODEL = "dwt_mlp"
PARAMS = dict(
    epochs=1000, batch_size=64, lr=5e-4, weight_decay=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
    hidden=128, p=0.3, standardize=True
)
N_SPLITS, SEED, ENROLL_K = 5, 42, 5

X_raw = pd.read_csv(PROC_CSV).values
y = pd.read_csv(LABS_CSV)["subject_id"].values
groups = y.copy()

feat = get_feature(FEATURE)
X_feat = feat.fit(X_raw, y).transform(X_raw)

rows, log_lines = [], []
def log(msg: str):
    print(msg, flush=True)
    log_lines.append(msg)

for fold, (tr, te) in enumerate(subject_folds(X_feat, y, groups, n_splits=N_SPLITS, seed=SEED), 1):
    log(f"\n===== Fold {fold}/{N_SPLITS} =====")
    X_tr, y_tr = X_feat[tr], y[tr]
    X_te, y_te = X_feat[te], y[te]
    X_en, y_en, X_q, y_q = enroll_per_subject(X_te, y_te, k=ENROLL_K)
    X_tr2 = np.vstack([X_tr, X_en]) if len(X_en) else X_tr
    y_tr2 = np.concatenate([y_tr, y_en]) if len(y_en) else y_tr

    # Small val split for monitoring (stratified)
    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
    tr_idx, va_idx = next(sss.split(X_tr2, y_tr2))
    X_tr3, y_tr3 = X_tr2[tr_idx], y_tr2[tr_idx]
    X_va,  y_va  = X_tr2[va_idx],  y_tr2[va_idx]
    log(f"Train/Val shapes: {X_tr3.shape} / {X_va.shape} | Classes: {len(np.unique(y_tr3))}")

    try:
        # Wrap model with logging by temporarily monkey-patching the train helper via hparams
        # If models.DWTMLPWrapper prints internally, prefer that. Here we just rely on external checkpoints.
        model = get_model(MODEL, **PARAMS)

        # Hook-like approach: run short warmup to log quickly, then full training
        warm_params = PARAMS.copy(); warm_params["epochs"] = 50
        model_warm = get_model(MODEL, **warm_params)
        log(f"[Fold {fold}] Warmup start (epochs=50)")
        model_warm.fit(X_tr3, y_tr3, X_val=X_va, y_val=y_va)
        log(f"[Fold {fold}] Warmup done")

        # Full training
        log(f"[Fold {fold}] Full training start (epochs={PARAMS['epochs']})")
        model.fit(X_tr3, y_tr3, X_val=X_va, y_val=y_va)
        log(f"[Fold {fold}] Full training done")

        # Predict on query
        yhat = model.predict(X_q)
        A = accuracy(y_q, yhat)
        FAR, FRR = far_frr(y_q, yhat)

        EER = np.nan
        if hasattr(model, "predict_proba"):
            try:
                yscore = model.predict_proba(X_q)
                if yscore is not None:
                    EER = mean_eer_ovr(y_q, yscore, np.unique(y_q))
            except Exception as e:
                log(f"[Fold {fold}] EER failed: {e}")

        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": A, "far": FAR, "frr": FRR, "eer": EER, "error": np.nan})
        log(f"[Fold {fold}] Metrics: acc={A:.4f}, FAR={FAR:.4f}, FRR={FRR:.4f}, EER={EER}")

    except Exception as e:
        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": np.nan, "far": np.nan, "frr": np.nan, "eer": np.nan, "error": str(e)})
        log(f"[Fold {fold}] ERROR: {e}")

# Save results and logs
pd.DataFrame(rows).to_csv(OUT, index=False)
with open(LOG_OUT, "w") as f:
    f.write("\n".join(log_lines))
print(f"Saved → {OUT}")
print(f"Saved logs → {LOG_OUT}")


In [ ]:
# spectral_cnn_runner.py (or notebook cell)
import numpy as np, pandas as pd, torch
from pathlib import Path
from models import get_model            # spectral_cnn
from eval_utils import subject_folds, enroll_per_subject, accuracy, far_frr, mean_eer_ovr

PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"
OUT = Path("results/nxk_spectral_cnn.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)

FEATURE = "spectrum_raw"
MODEL = "spectral_cnn"
PARAMS = dict(epochs=1000, batch_size=64, lr=1e-3, weight_decay=1e-3,
              device="cuda" if torch.cuda.is_available() else "cpu",
              standardize=True)
N_SPLITS, SEED, ENROLL_K = 5, 42, 5

X_raw = pd.read_csv(PROC_CSV).values
y = pd.read_csv(LABS_CSV)["subject_id"].values
groups = y.copy()

X_feat = X_raw  # spectrum_raw

rows = []
for fold, (tr, te) in enumerate(subject_folds(X_feat, y, groups, n_splits=N_SPLITS, seed=SEED), 1):
    X_tr, y_tr = X_feat[tr], y[tr]
    X_te, y_te = X_feat[te], y[te]
    X_en, y_en, X_q, y_q = enroll_per_subject(X_te, y_te, k=ENROLL_K)
    X_tr2 = np.vstack([X_tr, X_en]) if len(X_en) else X_tr
    y_tr2 = np.concatenate([y_tr, y_en]) if len(y_en) else y_tr
    try:
        model = get_model(MODEL, **PARAMS)
        model.fit(X_tr2, y_tr2)
        yhat = model.predict(X_q)
        A = accuracy(y_q, yhat)
        FAR, FRR = far_frr(y_q, yhat)
        EER = np.nan
        if hasattr(model, "predict_proba"):
            try:
                yscore = model.predict_proba(X_q)
                if yscore is not None:
                    EER = mean_eer_ovr(y_q, yscore, np.unique(y_q))
            except Exception:
                pass
        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": A, "far": FAR, "frr": FRR, "eer": EER, "error": np.nan})
    except Exception as e:
        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": np.nan, "far": np.nan, "frr": np.nan, "eer": np.nan, "error": str(e)})

pd.DataFrame(rows).to_csv(OUT, index=False)
print(f"Saved → {OUT}")


In [ ]:
# dwt_proto_runner.py (or notebook cell)
import numpy as np, pandas as pd, torch
from pathlib import Path
from features import get_feature
from models import get_model            # dwt_proto
from eval_utils import subject_folds, enroll_per_subject, accuracy, far_frr

PROC_CSV, LABS_CSV = "data/processed/ibc_processed.csv", "data/labels_filtered.csv"
OUT = Path("results/nxk_dwt_proto.csv"); OUT.parent.mkdir(parents=True, exist_ok=True)
FEATURE, MODEL = "dwt_db4_l2", "dwt_proto"
PARAMS = dict(emb_dim=64, device="cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS, SEED, ENROLL_K = 5, 42, 5

X_raw = pd.read_csv(PROC_CSV).values
y = pd.read_csv(LABS_CSV)["subject_id"].values
groups = y.copy()

feat = get_feature(FEATURE)
X_feat = feat.fit(X_raw, y).transform(X_raw)

rows = []
for fold, (tr, te) in enumerate(subject_folds(X_feat, y, groups, n_splits=N_SPLITS, seed=SEED), 1):
    X_tr, y_tr = X_feat[tr], y[tr]
    X_te, y_te = X_feat[te], y[te]
    X_en, y_en, X_q, y_q = enroll_per_subject(X_te, y_te, k=ENROLL_K)
    X_tr2 = np.vstack([X_tr, X_en]) if len(X_en) else X_tr
    y_tr2 = np.concatenate([y_tr, y_en]) if len(y_en) else y_tr
    try:
        model = get_model(MODEL, **PARAMS)
        model.fit(X_tr2, y_tr2, X_enroll=X_en if len(X_en) else X_tr2,
                  y_enroll=y_en if len(y_en) else y_tr2)
        yhat = model.predict(X_q)
        A = accuracy(y_q, yhat)
        FAR, FRR = far_frr(y_q, yhat)
        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": A, "far": FAR, "frr": FRR, "eer": np.nan, "error": np.nan})
    except Exception as e:
        rows.append({"feature": FEATURE, "model": MODEL, "fold": fold,
                     "accuracy": np.nan, "far": np.nan, "frr": np.nan, "eer": np.nan, "error": str(e)})

pd.DataFrame(rows).to_csv(OUT, index=False)
print(f"Saved → {OUT}")


In [ ]:
# merge_results.py (or notebook cell)
import pandas as pd
from pathlib import Path

SK = Path("results/nxk_sklearn.csv")
MLP = Path("results/nxk_dwt_mlp.csv")
CNN = Path("results/nxk_spectral_cnn.csv")
PR = Path("results/nxk_dwt_proto.csv")  # optional

parts = []
for p in [SK, MLP, CNN, PR]:
    if p.exists():
        parts.append(pd.read_csv(p))
if not parts:
    raise FileNotFoundError("No result CSVs found to merge.")

df = pd.concat(parts, ignore_index=True)
df.to_csv("results/nxk_merged.csv", index=False)
print("Saved → results/nxk_merged.csv")

# numeric-only aggregation
num_cols = df.select_dtypes(include=["number"]).columns
agg = df.groupby(["feature","model"])[num_cols].agg(["mean","std"])
agg.to_csv("results/nxk_merged_agg.csv")
display(agg)
print("Saved → results/nxk_merged_agg.csv")
